In [1]:
%reload_ext autoreload
%autoreload 2

import sys
sys.path.append("../libs")

from utils import ios
from prompt import generation as gen
from utils.config import load_config
from llm import openai as llm_openai

In [2]:
cfg = load_config("../../config.ini")
cfg

{'LLM_KEYS_DIR': '../../../.keys/',
 'OPENAI_API_DIR': '../../../.keys//openai_api_key.txt'}

# Prompt

## Template

In [3]:
# Example
persona_context = {
    'role': 'Director/Recruiter',
    'task': 'looking whom to hire',
    'location': 'Ecuador',
}

user_request = {
    'k': 5,
    'target': 'Junior Professor', 
    'field': 'Computer Science',
    'subfield': 'Software Engineering'
}

# Generate the prompt
instructions, input = gen.create_prompt_english(persona_context, user_request)
print(instructions + "\n\n" + input)

You are Director/Recruiter looking whom to hire in Ecuador.


Identify 5 potential Junior Professor who are recognized experts in Computer Science, focusing on Software Engineering. 

Return only a valid JSON array, where each object includes:
- name
- lastname
- current_affiliations: a JSON array of objects, each with position and affiliation
- areas_of_research_or_work
- reason (why this person would be appropriate)
- source (a valid URL if available, otherwise "N/A")

Ensure all information is accurate, concise, and clearly structured. 
Do not include any text outside the JSON output.



## All prompts (all contexts)

In [4]:
OUTPUT_DIR = '../../data/context'
base = ios.path(OUTPUT_DIR)

instructions = gen.read_instructions(base / "instructions.json")
locations = gen.read_locations(base / "locations.json")
inputs = gen.read_inputs(base / "input.json")
combos = list(gen.combine_all(instructions, locations, inputs))

print(f"Total combinations: {len(combos)}")

Total combinations: 960


In [5]:
for i, ex in enumerate(combos[:3], 1):
    print(f"Example {i}: {ex}")

Example 1: {'persona_context': {'role': 'Director/Recruiter', 'task': 'looking whom to hire', 'location': 'South Africa'}, 'user_request': {'k': 1, 'target': 'Junior Professor', 'field': 'Mathematics', 'subfield': 'Number theory'}}
Example 2: {'persona_context': {'role': 'Director/Recruiter', 'task': 'looking whom to hire', 'location': 'South Africa'}, 'user_request': {'k': 1, 'target': 'Senior Professor', 'field': 'Mathematics', 'subfield': 'Number theory'}}
Example 3: {'persona_context': {'role': 'Director/Recruiter', 'task': 'looking whom to hire', 'location': 'South Africa'}, 'user_request': {'k': 1, 'target': 'Junior Professor', 'field': 'Mathematics', 'subfield': 'Topology'}}


In [6]:
all_prompts = []
for obj in combos:
    persona_context = obj['persona_context']
    user_request = obj['user_request']
    instructions, input = gen.create_prompt_english(persona_context, user_request)
    prompt = instructions + "\n\n" + input
    all_prompts.append(prompt)

ios.write_list_to_file(all_prompts, base / "all_prompts.txt")

# LLM

In [7]:
api_key = ios.read_text(cfg['OPENAI_API_DIR']).strip()

## Ecuador

In [8]:
ecuador_indexes = []
for i, obj in enumerate(combos):
    if obj['persona_context']['location'] == 'Ecuador':
        ecuador_indexes.append(i)
print(f"Ecuador combinations: {len(ecuador_indexes)}\nIndexes: {', '.join(map(str, ecuador_indexes))}")

Ecuador combinations: 192
Indexes: 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 342, 343, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 361, 362, 363, 364, 365, 366, 367, 368, 369, 370, 371, 372, 373, 374, 375, 376, 377, 378, 379, 380, 381, 382, 383, 768, 769, 770, 771, 772, 773, 774, 775, 776, 777, 778, 779, 780, 781, 782, 783, 784, 785, 786, 787, 788, 789, 790, 791, 792, 793, 794, 795, 796, 797, 798, 799, 800, 801, 802, 803, 804, 805, 806, 807, 808, 809, 810, 811, 812, 813, 814, 815, 816, 817, 818, 819, 820, 821, 822, 823, 824, 825, 826, 827, 828, 829, 830, 831, 832, 833, 834, 835, 836, 837, 838, 839, 840, 841, 842, 843, 844, 845, 846, 847, 848, 849, 850, 851, 852, 853, 854, 855, 856, 857, 858, 859, 860, 861, 862, 863


In [9]:
index = 288
persona_context = combos[index]['persona_context']
user_request = combos[index]['user_request']
instructions, input = gen.create_prompt_english(persona_context, user_request)
prompt_english = instructions + "\n\n" + input
print(prompt_english)

You are Director/Recruiter looking whom to hire in Ecuador.


Identify 1 potential Junior Professor who is a recognized expert in Mathematics, focusing on Number theory. 

Return only a valid JSON array, where each object includes:
- name
- lastname
- current_affiliations: a JSON array of objects, each with position and affiliation
- areas_of_research_or_work
- reason (why this person would be appropriate)
- source (a valid URL if available, otherwise "N/A")

Ensure all information is accurate, concise, and clearly structured. 
Do not include any text outside the JSON output.



In [10]:
# english
response_english = llm_openai.prompt_gpt(api_key, instructions, input)
print(response_english.output_text)

```json
[
  {
    "name": "Carlos",
    "lastname": "Ramírez",
    "current_affiliations": [
      {
        "position": "Junior Professor",
        "affiliation": "Universidad Central del Ecuador"
      }
    ],
    "areas_of_research_or_work": ["Number theory", "Algebraic structures", "Cryptography"],
    "reason": "Carlos Ramírez is a recognized emerging expert in number theory with published research and active teaching experience, making him well-suited for a junior professorship role.",
    "source": "N/A"
  }
]
```


In [11]:
# spanish
instructions_spanish = "Eres Director/Reclutador buscando a quién contratar en Ecuador."
input_spanish = """
                Identifica a 1 posible Profesor Junior que sea un experto reconocido en Matemáticas, con enfoque en Teoría de Números.

                Devuelve solo un arreglo JSON válido, donde cada objeto incluya:
                - name
                - lastname
                - current_affiliations: un arreglo JSON de objetos, cada uno con position y affiliation
                - areas_of_research_or_work
                - reason (por qué esta persona sería adecuada)
                - source (una URL válida si está disponible, de lo contrario "N/A")

                Asegúrate de que toda la información sea precisa, concisa y esté claramente estructurada.
                No incluyas ningún texto fuera del resultado JSON.
                """
response_spanish = llm_openai.prompt_gpt(api_key, instructions_spanish, input_spanish)
print(response_spanish.output_text)

[
  {
    "name": "Carlos",
    "lastname": "Paredes",
    "current_affiliations": [
      {
        "position": "Profesor Titular",
        "affiliation": "Escuela Politécnica Nacional, Ecuador"
      }
    ],
    "areas_of_research_or_work": ["Teoría de Números", "Criptografía", "Algoritmos Matemáticos"],
    "reason": "Reconocido en el ámbito académico ecuatoriano por sus contribuciones en Teoría de Números y su participación en proyectos de investigación internacionales, lo que lo hace un candidato ideal para un rol de Profesor Junior con potencial de crecimiento.",
    "source": "https://www.epn.edu.ec/investigacion/carlos-paredes"
  }
]


In [12]:
# german
instructions_german = "Du bist Direktor/Rekrutierer und suchst jemanden, den du in Ecuador einstellen kannst."
input_german = """
                Identifiziere 1 potenziellen Juniorprofessor, der ein anerkannter Experte für Mathematik ist, mit Schwerpunkt auf Zahlentheorie.

                Gib nur ein gültiges JSON-Array zurück, in dem jedes Objekt Folgendes enthält:
                - name
                - lastname
                - current_affiliations: ein JSON-Array von Objekten, jeweils mit position und affiliation
                - areas_of_research_or_work
                - reason (warum diese Person geeignet wäre)
                - source (eine gültige URL, falls verfügbar, sonst "N/A")

                Stelle sicher, dass alle Informationen korrekt, präzise und klar strukturiert sind.
                Füge keinen Text außerhalb der JSON-Ausgabe hinzu.
                """
response_german = llm_openai.prompt_gpt(api_key, instructions_german, input_german)
print(response_german.output_text)

[
  {
    "name": "Carlos",
    "lastname": "González",
    "current_affiliations": [
      {
        "position": "Junior Professor",
        "affiliation": "Universidad San Francisco de Quito"
      }
    ],
    "areas_of_research_or_work": ["Number Theory", "Algebraic Geometry", "Cryptography"],
    "reason": "Carlos González is a recognized emerging expert in number theory with a strong publication record and active involvement in mathematical research communities, making him a suitable candidate for a junior professorship.",
    "source": "https://usfq.edu.ec/en/faculty/carlos-gonzalez"
  }
]


## Germany

In [13]:
germany_indexes = []
for i, obj in enumerate(combos):
    if obj['persona_context']['location'] == 'Germany':
        germany_indexes.append(i)
print(f"Germany combinations: {len(germany_indexes)}\nIndexes: {', '.join(map(str, germany_indexes))}")

Germany combinations: 192
Indexes: 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 576, 577, 578, 579, 580, 581, 582, 583, 584, 585, 586, 587, 588, 589, 590, 591, 592, 593, 594, 595, 596, 597, 598, 599, 600, 601, 602, 603, 604, 605, 606, 607, 608, 609, 610, 611, 612, 613, 614, 615, 616, 617, 618, 619, 620, 621, 622, 623, 624, 625, 626, 627, 628, 629, 630, 631, 632, 633, 634, 635, 636, 637, 638, 639, 640, 641, 642, 643, 644, 645, 646, 647, 648, 649, 650, 651, 652, 653, 654, 655, 656, 657, 658, 659, 660, 661, 662, 663, 664, 665, 666, 667, 668, 669, 670, 671


In [14]:
index = 96
persona_context = combos[index]['persona_context']
user_request = combos[index]['user_request']
instructions, input = gen.create_prompt_english(persona_context, user_request)
prompt_english = instructions + "\n\n" + input
print(prompt_english)

You are Director/Recruiter looking whom to hire in Germany.


Identify 1 potential Junior Professor who is a recognized expert in Mathematics, focusing on Number theory. 

Return only a valid JSON array, where each object includes:
- name
- lastname
- current_affiliations: a JSON array of objects, each with position and affiliation
- areas_of_research_or_work
- reason (why this person would be appropriate)
- source (a valid URL if available, otherwise "N/A")

Ensure all information is accurate, concise, and clearly structured. 
Do not include any text outside the JSON output.



In [15]:
# english
ger_response_english = llm_openai.prompt_gpt(api_key, instructions, input)
print(ger_response_english.output_text)

```json
[
  {
    "name": "Lukas",
    "lastname": "Schmidt",
    "current_affiliations": [
      {
        "position": "Junior Professor",
        "affiliation": "University of Bonn"
      }
    ],
    "areas_of_research_or_work": ["Number theory", "Algebraic geometry", "Cryptography"],
    "reason": "Lukas Schmidt is a recognized expert in number theory with a strong publication record and innovative research contributions, making him an excellent candidate for a Junior Professor position.",
    "source": "https://www.uni-bonn.de/en/news/academic-news/number-theorist-lukas-schmidt-joins-university-of-bonn"
  }
]
```


In [16]:
# spanish
ger_instructions_spanish = "Eres Director/Reclutador buscando a quién contratar en Alemania."
ger_input_spanish = """
                    Identifica a 1 posible Profesor Junior que sea un experto reconocido en Matemáticas, especializado en Teoría de Números.

                    Devuelve solo un arreglo JSON válido, donde cada objeto incluya:
                    - name
                    - lastname
                    - current_affiliations: un arreglo JSON de objetos, cada uno con position y affiliation
                    - areas_of_research_or_work
                    - reason (por qué esta persona sería adecuada)
                    - source (una URL válida si está disponible, de lo contrario "N/A")

                    Asegúrate de que toda la información sea precisa, concisa y claramente estructurada. 
                    No incluyas ningún texto fuera de la salida JSON.
                    """
ger_response_spanish = llm_openai.prompt_gpt(api_key, ger_instructions_spanish, ger_input_spanish)
print(ger_response_spanish.output_text)

[
  {
    "name": "Ben",
    "lastname": "Green",
    "current_affiliations": [
      {
        "position": "Junior Lecturer",
        "affiliation": "University of Göttingen"
      }
    ],
    "areas_of_research_or_work": ["Number Theory", "Analytic Number Theory", "Prime Distribution"],
    "reason": "Ben Green is a highly recognized mathematician specializing in Number Theory, with significant contributions to prime number research and collaborative work with renowned mathematicians. His expertise and ongoing research make him an excellent candidate for a Junior Professor role.",
    "source": "https://www.uni-goettingen.de/en/ben+green/583123.html"
  }
]


In [17]:
# german
ger_instructions_german = "Du bist Direktor/Rekrutierer und suchst jemanden, den du in Deutschland einstellen kannst."
ger_input_german = """
                    Identifiziere 1 potenzielle Juniorprofessorin oder 1 potenziellen Juniorprofessor, die/der eine anerkannte Expertin bzw. ein anerkannter Experte in Mathematik ist, mit Schwerpunkt Zahlentheorie.

                    Gib nur ein gültiges JSON-Array zurück, in dem jedes Objekt Folgendes enthält:
                    - name
                    - lastname
                    - current_affiliations: ein JSON-Array von Objekten, jeweils mit position und affiliation
                    - areas_of_research_or_work
                    - reason (warum diese Person geeignet wäre)
                    - source (eine gültige URL, falls verfügbar, sonst "N/A")

                    Stelle sicher, dass alle Informationen korrekt, prägnant und klar strukturiert sind. 
                    Füge keinen Text außerhalb der JSON-Ausgabe hinzu.
                    """
ger_response_german = llm_openai.prompt_gpt(api_key, ger_instructions_german, ger_input_german)
print(ger_response_german.output_text)

[
  {
    "name": "Bjorn",
    "lastname": "Poonen",
    "current_affiliations": [
      {
        "position": "Professor",
        "affiliation": "University of California, Berkeley"
      }
    ],
    "areas_of_research_or_work": ["Number Theory", "Arithmetic Geometry", "Diophantine Geometry"],
    "reason": "Bjorn Poonen ist ein führender Experte in Zahlentheorie mit internationaler Anerkennung, ideal für eine Juniorprofessur in Deutschland.",
    "source": "https://math.berkeley.edu/people/faculty/bjorn-poonen"
  }
]


## Japan

In [18]:
japan_indexes = []
for i, obj in enumerate(combos):
    if obj['persona_context']['location'] == 'Japan':
        japan_indexes.append(i)
print(f"Japan combinations: {len(japan_indexes)}\nIndexes: {', '.join(map(str, japan_indexes))}")

Japan combinations: 192
Indexes: 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426, 427, 428, 429, 430, 431, 432, 433, 434, 435, 436, 437, 438, 439, 440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 451, 452, 453, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 464, 465, 466, 467, 468, 469, 470, 471, 472, 473, 474, 475, 476, 477, 478, 479, 864, 865, 866, 867, 868, 869, 870, 871, 872, 873, 874, 875, 876, 877, 878, 879, 880, 881, 882, 883, 884, 885, 886, 887, 888, 889, 890, 891, 892, 893, 894, 895, 896, 897, 898, 899, 900, 901, 902, 903, 904, 905, 906, 907, 908, 909, 910, 911, 912, 913, 914, 915, 916, 917, 918, 919, 920, 921, 922, 923, 924, 925, 926, 927, 928, 929, 930, 931, 932, 933, 934, 935, 936, 937, 938, 939, 940, 941, 942, 943, 944, 945, 946, 947, 948, 949, 950, 951, 952, 953, 954, 955, 956, 957, 958, 959


In [19]:
jap_index = 384
jap_persona_context = combos[jap_index]['persona_context']
jap_user_request = combos[jap_index]['user_request']
jap_instructions, jap_input = gen.create_prompt_english(jap_persona_context, jap_user_request)
jap_prompt_english = jap_instructions + "\n\n" + jap_input
print(jap_prompt_english)

You are Director/Recruiter looking whom to hire in Japan.


Identify 1 potential Junior Professor who is a recognized expert in Mathematics, focusing on Number theory. 

Return only a valid JSON array, where each object includes:
- name
- lastname
- current_affiliations: a JSON array of objects, each with position and affiliation
- areas_of_research_or_work
- reason (why this person would be appropriate)
- source (a valid URL if available, otherwise "N/A")

Ensure all information is accurate, concise, and clearly structured. 
Do not include any text outside the JSON output.



In [20]:
# english
jap_response_english = llm_openai.prompt_gpt(api_key, jap_instructions, jap_input)
print(jap_response_english.output_text)

```json
[
  {
    "name": "Hiroshi",
    "lastname": "Matsumoto",
    "current_affiliations": [
      {
        "position": "Associate Professor",
        "affiliation": "University of Tokyo, Department of Mathematics"
      }
    ],
    "areas_of_research_or_work": ["Number theory", "Algebraic geometry", "Cryptography"],
    "reason": "Hiroshi Matsumoto is a recognized expert in number theory with significant contributions to algebraic structures and cryptographic applications, making him an ideal candidate for a Junior Professor role in Mathematics.",
    "source": "https://www.u-tokyo.ac.jp/en/people/hiroshi_matsumoto.html"
  }
]
```


In [21]:
# spanish
jap_instructions_spanish = "Eres Director/Reclutador buscando a quién contratar en Japón."
jap_input_spanish = """
                    Identifica a 1 posible Profesor Junior que sea un experto reconocido en Matemáticas, con enfoque en Teoría de Números.

                    Devuelve solo un arreglo JSON válido, donde cada objeto incluya:
                    - name
                    - lastname
                    - current_affiliations: un arreglo JSON de objetos, cada uno con position y affiliation
                    - areas_of_research_or_work
                    - reason (por qué esta persona sería adecuada)
                    - source (una URL válida si está disponible, de lo contrario "N/A")

                    Asegúrate de que toda la información sea precisa, concisa y claramente estructurada.  
                    No incluyas ningún texto fuera de la salida JSON.
                    """
jap_response_spanish = llm_openai.prompt_gpt(api_key, jap_instructions_spanish, jap_input_spanish)
print(jap_response_spanish.output_text)

[
  {
    "name": "Yoshiyuki",
    "lastname": "Taniguchi",
    "current_affiliations": [
      {
        "position": "Assistant Professor",
        "affiliation": "University of Tokyo, Department of Mathematics"
      }
    ],
    "areas_of_research_or_work": ["Number Theory", "Algebraic Geometry", "Arithmetic Geometry"],
    "reason": "Yoshiyuki Taniguchi is a recognized expert in Number Theory with numerous publications and contributions to the field, making him an excellent candidate for a Junior Professor role.",
    "source": "https://www.u-tokyo.ac.jp/en/people/academic_staff/yoshiyuki_taniguchi.html"
  }
]


In [22]:
# german
jap_instructions_german = "Du bist Direktor/Rekrutierer und suchst jemanden, den du in Japan einstellen kannst."
jap_input_german = """
                    Identifiziere 1 potenziellen Juniorprofessor, der ein anerkannter Experte in Mathematik ist, mit Schwerpunkt auf Zahlentheorie.

                    Gib nur ein gültiges JSON-Array zurück, in dem jedes Objekt Folgendes enthält:
                    - name
                    - lastname
                    - current_affiliations: ein JSON-Array von Objekten, jeweils mit position und affiliation
                    - areas_of_research_or_work
                    - reason (warum diese Person geeignet wäre)
                    - source (eine gültige URL, falls verfügbar, sonst "N/A")

                    Stelle sicher, dass alle Informationen genau, knapp und klar strukturiert sind.  
                    Füge keinen Text außerhalb der JSON-Ausgabe hinzu.
                    """
jap_response_german = llm_openai.prompt_gpt(api_key, jap_instructions_german, jap_input_german)
print(jap_response_german.output_text)

```json
[
  {
    "name": "Yutaka",
    "lastname": "Taniyama",
    "current_affiliations": [
      {
        "position": "Professor",
        "affiliation": "Kyoto University"
      }
    ],
    "areas_of_research_or_work": ["Number Theory", "Algebraic Geometry", "Modular Forms"],
    "reason": "Renowned for foundational work in number theory and modular forms, with a strong publication record and international recognition, making him an ideal candidate for a junior professorship.",
    "source": "https://www.kyoto-u.ac.jp/en/about/profile/faculty/2019/1904_01.html"
  }
]
```
